# 23CSE301 ML Capstone — EV Charging Stations

## Notebook Structure
1. **Dataset Understanding**
2. **Preprocessing**
3. **REGRESSIONS — Part 1**
4. **REGRESSIONS — Part 2**
5. **CLASSIFIER**

Preprocessing is kept separate so the prepared data can be reused by all models.

## 0. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = "EV_Charging_Stations_Feb82024.xlsx"
SHEET_NAME = "Raw"

REG_TARGET = "EV Level2 EVSE Num"
CLF_TARGET = "Access Code"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Dataset Understanding

The dataset contains EV charging-station records. The proposed regression target is **`EV Level2 EVSE Num`** and the proposed classification target is **`Access Code`** (`public` / `private`).

In [2]:
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


Shape: (65134, 74)
Rows: 65134
Columns: 74


,Fuel Type Code,Station Name,Street Address,Intersection Directions,City,State,ZIP,Plus4,Station Phone,Status Code,Expected Date,Groups With Access Code,Access Days Time,Cards Accepted,BD Blends,NG Fill Type Code,NG PSI,EV Level1 EVSE Num,EV Level2 EVSE Num,EV DC Fast Count,EV Other Info,EV Network,EV Network Web,Geocode Status,Latitude,Longitude,Date Last Confirmed,ID,Updated At,Owner Type Code,Federal Agency ID,Federal Agency Name,Open Date,Hydrogen Status Link,NG Vehicle Class,LPG Primary,E85 Blender Pump,EV Connector Types,Country,Intersection Directions (French),Access Days Time (French),BD Blends (French),Groups With Access Code (French),Hydrogen Is Retail,Access Code,Access Detail Code,Federal Agency Code,Facility Type,CNG Dispenser Num,CNG On-Site Renewable Source,CNG Total Compression Capacity,CNG Storage Capacity,LNG On-Site Renewable Source,E85 Other Ethanol Blends,EV Pricing,EV Pricing (French),LPG Nozzle Types,Hydrogen Pressures,Hydrogen Standards,CNG Fill Type Code,CNG PSI,CNG Vehicle Class,LNG Vehicle Class,EV On-Site Renewable Source,Restricted Access,RD Blends,RD Blends (French),RD Blended with Biodiesel,RD Maximum Biodiesel Level,NPS Unit Name,CNG Station Sells Renewable Natural Gas,LNG Station Sells Renewable Natural Gas,Maximum Vehicle Class,EV Workplace Charging
0,ELEC,LADWP - Truesdale Center,11797 Truesdale St,NaN,Sun Valley,CA,91352,NaN,NaN,E,NaN,Private,Fleet use only,NaN,NaN,NaN,NaN,NaN,57.0,2.0,NaN,SHELL_RECHARGE,https://shellrecharge.com/en-us/solutions,GPS,34.248319,-118.387971,2023-09-14,1517,2024-01-31 22:07:01 UTC,LG,NaN,NaN,1999-10-15,NaN,NaN,NaN,NaN,CHADEMO J1772 J1772COMBO,US,NaN,NaN,NaN,Privé,NaN,private,NaN,NaN,UTILITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,ELEC,Los Angeles Convention Center,1201 S Figueroa St,West hall and South hall,Los Angeles,CA,90015,NaN,213-741-1151,E,NaN,Public,5:30am-9pm; pay lot,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,Non-Networked,NaN,GPS,34.040539,-118.271387,2023-01-10,1523,2023-02-14 15:54:11 UTC,P,NaN,NaN,1995-08-30,NaN,NaN,NaN,NaN,J1772,US,NaN,NaN,NaN,Public,NaN,public,NaN,NaN,PARKING_GARAGE,NaN,NaN,NaN,NaN,NaN,NaN,Free; parking fee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LD,0.0
2,ELEC,LADWP - John Ferraro Building,111 N Hope St,Across Hope,Los Angeles,CA,90012,NaN,NaN,E,NaN,Private,For fleet and employee use only,NaN,NaN,NaN,NaN,NaN,338.0,12.0,NaN,Non-Networked,NaN,GPS,34.059133,-118.248589,2023-09-14,1525,2024-01-31 22:07:01 UTC,LG,NaN,NaN,1999-10-15,NaN,NaN,NaN,NaN,CHADEMO J1772 J1772COMBO,US,NaN,NaN,NaN,Privé,NaN,private,NaN,NaN,UTILITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LD,1.0
3,ELEC,LADWP - Haynes Power Plant,6801 E 2nd St,NaN,Long Beach,CA,90803,NaN,NaN,E,NaN,Private,Fleet use only,NaN,NaN,NaN,NaN,NaN,19.0,1.0,NaN,Non-Networked,NaN,GPS,33.759802,-118.096665,2024-01-09,1531,2024-01-31 22:07:01 UTC,LG,NaN,NaN,2018-05-01,NaN,NaN,NaN,NaN,CHADEMO J1772 J1772COMBO,US,NaN,NaN,NaN,Privé,NaN,private,NaN,NaN,UTILITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,ELEC,LADWP - Harbor Generating Station,161 N Island Ave,At B St,Wilmington,CA,90744,NaN,NaN,E,NaN,Private,Fleet use only,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,Non-Networked,NaN,200-8,33.770508,-118.265628,2024-01-09,1552,2024-01-31 22:07:01 UTC,LG,NaN,NaN,1999-10-15,NaN,NaN,NaN,NaN,J1772,US,NaN,NaN,NaN,Privé,NaN,private,NaN,NaN,UTILITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


### 1.1 Structure and data types

In [3]:
structure = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})
display(structure)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


,Column,Data Type,Non-Null,Null,Unique Values
0,Fuel Type Code,object,65134,0,1
1,Station Name,object,65131,3,62508
2,Street Address,object,65099,35,45040
3,Intersection Directions,object,2361,62773,2101
4,City,object,65129,5,6494
5,State,object,65120,14,52
6,ZIP,object,65133,1,11104
7,Plus4,float64,0,65134,0
8,Station Phone,object,61483,3651,9049
9,Status Code,object,65134,0,1


Numeric columns: 42
Categorical columns: 30


### 1.2 Basic checks

In [4]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing Count"))

print("\nNumeric summary:")
display(df[numeric_cols].describe().T)


Duplicate rows: 0

Missing values:


,Missing Count
CNG PSI,65134
CNG Total Compression Capacity,65134
E85 Blender Pump,65134
LPG Primary,65134
NG Vehicle Class,65134
Hydrogen Status Link,65134
BD Blends (French),65134
CNG Vehicle Class,65134
Hydrogen Is Retail,65134
RD Blends,65134



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
Plus4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Expected Date,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BD Blends,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NG Fill Type Code,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NG PSI,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EV Level1 EVSE Num,667.0,4.379310,8.532664,1.000000,1.000000,2.000000,4.000000,90.000000
EV Level2 EVSE Num,56677.0,2.420382,3.273703,1.000000,2.000000,2.000000,2.000000,338.000000
EV DC Fast Count,9364.0,4.206002,5.058703,1.000000,1.000000,2.000000,6.000000,84.000000
Latitude,65134.0,37.816967,5.040287,-41.354100,34.040619,38.503573,41.488183,64.852466
Longitude,65134.0,-96.670801,19.528462,-164.848855,-117.927948,-92.807549,-78.963729,122.373276


In [5]:
print("Regression target:", REG_TARGET)
display(df[REG_TARGET].describe())

print("\nClassification target:", CLF_TARGET)
display(df[CLF_TARGET].value_counts(dropna=False))


Regression target: EV Level2 EVSE Num


count    56677.000000
mean         2.420382
std          3.273703
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max        338.000000
Name: EV Level2 EVSE Num, dtype: float64


Classification target: Access Code


Access Code
public     61308
private     3826
Name: count, dtype: int64

## 2. Preprocessing

### Order
1. Remove exact duplicates
2. Remove rows with missing target
3. Select useful predictors and avoid leakage
4. Train/test split
5. Detect numeric outliers using **IQR**
6. Cap outliers using training-derived IQR limits
7. Fill numeric missing values using **KNN Imputer**
8. Fill categorical missing values
9. **One-Hot Encode** categorical variables
10. **Standardize** numeric/encoded features
11. Apply **PCA** for dimensionality reduction

> Preprocessing is fitted on the training data only to avoid data leakage.

### 2.1 Basic cleaning and feature engineering

In [6]:
data = df.copy()

before = len(data)
data = data.drop_duplicates().copy()
print("Duplicates removed:", before - len(data))

if "Open Date" in data.columns:
    open_date = pd.to_datetime(data["Open Date"], errors="coerce")
    data["Open Year"] = open_date.dt.year
    data["Station Age"] = 2024 - data["Open Year"]

if "EV Connector Types" in data.columns:
    data["Num Connector Types"] = (
        data["EV Connector Types"].fillna("").astype(str).str.strip()
        .apply(lambda x: 0 if not x else len(set(x.split())))
    )

if "EV Network" in data.columns:
    data["Is Networked"] = (
        data["EV Network"].fillna("Non-Networked").astype(str).str.lower()
        .ne("non-networked").astype(int)
    )

if "EV Pricing" in data.columns:
    data["Has Pricing"] = data["EV Pricing"].notna().astype(int)

if "EV Workplace Charging" in data.columns:
    data["Has Workplace Charging"] = (
        pd.to_numeric(data["EV Workplace Charging"], errors="coerce").fillna(0) > 0
    ).astype(int)

print("Shape after cleaning:", data.shape)


Duplicates removed: 0
Shape after cleaning: (65134, 80)


### 2.2 Feature selection

In [ ]:
# Regression: remove target and direct charger-count fields to prevent leakage
reg_features = [
    c for c in [
        "Latitude", "Longitude", "Open Year", "Station Age",
        "Num Connector Types", "Is Networked", "Has Pricing",
        "Has Workplace Charging", "Access Code", "State", "Facility Type",
        "EV Network"
    ] if c in data.columns and c not in {
        REG_TARGET, "EV Level1 EVSE Num", "EV DC Fast Count"
    }]

# Classification: remove target and target-defining field
clf_features = [
    c for c in [
        "Latitude", "Longitude", "Open Year", "Station Age",
        "Num Connector Types", "Is Networked", "Has Pricing",
        "Has Workplace Charging", "State", "Facility Type",
        "EV Network", "EV Connector Types"
    ] if c in data.columns and c not in {CLF_TARGET, "Access Detail Code"}]

print("Regression features:", reg_features)
print("\nClassification features:", clf_features)


### 2.3 Train/test split

In [ ]:
reg_df = data[reg_features + [REG_TARGET]].dropna(subset=[REG_TARGET]).copy()
X_reg = reg_df[reg_features]
y_reg = pd.to_numeric(reg_df[REG_TARGET], errors="coerce")

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

clf_df = data[clf_features + [CLF_TARGET]].dropna(subset=[CLF_TARGET]).copy()
X_clf = clf_df[clf_features]
y_clf = clf_df[CLF_TARGET].astype(str)

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_clf
)

print("Regression:", X_reg_train.shape, X_reg_test.shape)
print("Classification:", X_clf_train.shape, X_clf_test.shape)


### 2.4 Outlier detection — IQR

**IQR = Q3 − Q1**

Potential outlier = value `< Q1 − 1.5×IQR` or `> Q3 + 1.5×IQR`.

In [ ]:
def iqr_report(X, numeric_columns):
    rows = []
    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = ((s < low) | (s > high)).sum()
        rows.append({
            "Feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "Lower Bound": low, "Upper Bound": high, "Outliers": int(count)
        })
    return pd.DataFrame(rows)

reg_num = X_reg_train.select_dtypes(include=np.number).columns.tolist()
clf_num = X_clf_train.select_dtypes(include=np.number).columns.tolist()

print("Regression outliers")
display(iqr_report(X_reg_train, reg_num))

print("Classification outliers")
display(iqr_report(X_clf_train, clf_num))


### 2.5 IQR outlier capping

In [ ]:
def fit_iqr_caps(X, numeric_columns):
    caps = {}
    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        caps[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return caps

def apply_iqr_caps(X, caps):
    X = X.copy()
    for col, (low, high) in caps.items():
        X[col] = pd.to_numeric(X[col], errors="coerce").clip(low, high)
    return X

reg_caps = fit_iqr_caps(X_reg_train, reg_num)
clf_caps = fit_iqr_caps(X_clf_train, clf_num)

X_reg_train = apply_iqr_caps(X_reg_train, reg_caps)
X_reg_test = apply_iqr_caps(X_reg_test, reg_caps)
X_clf_train = apply_iqr_caps(X_clf_train, clf_caps)
X_clf_test = apply_iqr_caps(X_clf_test, clf_caps)

print("IQR capping completed.")


### 2.6 KNN Imputation + One-Hot Encoding + Standardization

In [ ]:
reg_num = X_reg_train.select_dtypes(include=np.number).columns.tolist()
reg_cat = X_reg_train.select_dtypes(exclude=np.number).columns.tolist()

clf_num = X_clf_train.select_dtypes(include=np.number).columns.tolist()
clf_cat = X_clf_train.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([
            ("knn_imputer", KNNImputer(n_neighbors=5)),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols)
    ])

reg_preprocessor = make_preprocessor(reg_num, reg_cat)
clf_preprocessor = make_preprocessor(clf_num, clf_cat)

print("Regression numeric:", reg_num)
print("Regression categorical:", reg_cat)
print("\nClassification numeric:", clf_num)
print("Classification categorical:", clf_cat)


### 2.7 Transform data

In [ ]:
X_reg_train_p = reg_preprocessor.fit_transform(X_reg_train)
X_reg_test_p = reg_preprocessor.transform(X_reg_test)

X_clf_train_p = clf_preprocessor.fit_transform(X_clf_train)
X_clf_test_p = clf_preprocessor.transform(X_clf_test)

print("Regression processed shape:", X_reg_train_p.shape)
print("Classification processed shape:", X_clf_train_p.shape)


### 2.8 PCA

In [15]:
# Retain 95% of the variance
pca_reg = PCA(n_components=0.95, svd_solver="full")
pca_clf = PCA(n_components=0.95, svd_solver="full")

X_reg_train_pca = pca_reg.fit_transform(X_reg_train_p)
X_reg_test_pca = pca_reg.transform(X_reg_test_p)

X_clf_train_pca = pca_clf.fit_transform(X_clf_train_p)
X_clf_test_pca = pca_clf.transform(X_clf_test_p)

print("Regression PCA shape:", X_reg_train_pca.shape)
print("Classification PCA shape:", X_clf_train_pca.shape)
print("Regression variance retained:", round(pca_reg.explained_variance_ratio_.sum(), 4))
print("Classification variance retained:", round(pca_clf.explained_variance_ratio_.sum(), 4))


Regression PCA shape: (45341, 37)
Classification PCA shape: (52107, 45)
Regression variance retained: 0.9513
Classification variance retained: 0.9509


## 3. REGRESSIONS — Part 1

### Models to fill later
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet
5. Polynomial Regression

**Metrics:** R², RMSE, MAE

Use:
`X_reg_train_pca`, `X_reg_test_pca`, `y_reg_train`, `y_reg_test`

In [14]:
linear_model = LinearRegression()

linear_model.fit(
    X_reg_train_pca,
    y_reg_train
)

y_pred_linear = linear_model.predict(
    X_reg_test_pca
)

r2_linear = r2_score(y_reg_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_reg_test, y_pred_linear))
mae_linear = mean_absolute_error(y_reg_test, y_pred_linear)

print("LINEAR REGRESSION")
print("-" * 40)
print(f"R²   : {r2_linear:.4f}")
print(f"RMSE : {rmse_linear:.4f}")
print(f"MAE  : {mae_linear:.4f}")


plt.figure(figsize=(8, 6))

plt.scatter(
    y_reg_test,
    y_pred_linear,
    alpha=0.5
)

min_value = min(y_reg_test.min(), y_pred_linear.min())
max_value = max(y_reg_test.max(), y_pred_linear.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title("Linear Regression - Actual vs Predicted")
plt.xlabel("Actual EV Level2 EVSE Num")
plt.ylabel("Predicted EV Level2 EVSE Num")

plt.tight_layout()
plt.show()

residuals_linear = y_reg_test - y_pred_linear

plt.figure(figsize=(8, 5))

plt.scatter(
    y_pred_linear,
    residuals_linear,
    alpha=0.5
)

plt.axhline(
    0,
    linestyle="--"
)

plt.title("Linear Regression - Residual Plot")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")

plt.tight_layout()
plt.show()


NameError: name 'LinearRegression' is not defined

## 4. REGRESSIONS — Part 2

### Models to fill later
6. Decision Tree Regressor
7. Random Forest Regressor
8. Gradient Boosting Regressor
9. Support Vector Regressor (SVR)
10. K-Nearest Neighbors Regressor

**Metrics:** R², RMSE, MAE

Use:
`X_reg_train_pca`, `X_reg_test_pca`, `y_reg_train`, `y_reg_test`

In [ ]:
# REGRESSIONS — PART 2
# 6. Decision Tree Regressor
# 7. Random Forest Regressor
# 8. Gradient Boosting Regressor
# 9. Support Vector Regressor
# 10. K-Nearest Neighbors Regressor

# Add model -> fit -> predict -> evaluate code here.


## 5. CLASSIFIER

### Models to fill later
1. Logistic Regression
2. K-Nearest Neighbors
3. Gaussian Naive Bayes
4. Decision Tree Classifier
5. Support Vector Classifier (SVC)

**Metrics:** Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC

Use:
`X_clf_train_pca`, `X_clf_test_pca`, `y_clf_train`, `y_clf_test`

In [ ]:
# CLASSIFIER
# 1. Logistic Regression
# 2. K-Nearest Neighbors
# 3. Gaussian Naive Bayes
# 4. Decision Tree Classifier
# 5. Support Vector Classifier

# Add model -> fit -> predict -> evaluate code here.


## Preprocessing Checklist

- Dataset structure and data types
- Missing-value inspection
- Duplicate removal
- Target definition
- Target leakage control
- Train/test split
- IQR outlier detection
- IQR outlier capping
- KNN imputation for numeric values
- Categorical imputation
- One-hot encoding
- Standardization
- PCA
- Regression shells
- Classifier shell
